# [Notebook]Unihack26_Synthetic_Ranker

Trainable reference ranking and evidence sufficiency on newly generated synthetic carton-face images. Frozen original DINOv2 ViT-S/14 (Apache-2.0), SIFT geometry, three training seeds, object-disjoint selection/calibration/test. No Parcel3D renders/textures used. No claim about real-parcel accuracy. Output remains a research candidate pending real-box validation.

In [ ]:
from pathlib import Path
import sys, json
SOURCES = {'ml/__init__.py': '', 'ml/data/__init__.py': '', 'ml/data/labels.py': '"""Validate pair supervision and detect split leakage before feature extraction."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport re\nfrom collections import defaultdict\n\nSPLITS = ("train", "validation", "test")\nCORRESPONDENCE = {"MATCH", "WRONG_REGION", "WRONG_OBJECT", "UNKNOWN"}\nSUFFICIENCY = {"SUFFICIENT", "INSUFFICIENT", "UNKNOWN"}\nCHANGE = {"CHANGED", "UNCHANGED", "UNKNOWN"}\nSHA256 = re.compile(r"^[a-f0-9]{64}$")\nREASONS = {"BLUR", "GLARE", "OCCLUSION", "ROI_OUTSIDE_OVERLAP", "LOW_RESOLUTION",\n           "VIEW_MISMATCH", "NO_REFERENCE", "UNKNOWN"}\n\n\ndef locked_split(lineages: list[str], seed: str = "unihack26-v1") -> dict:\n    """Split a fixed, deduplicated corpus 60/20/20. Persist; never refit mid-run."""\n    if len(lineages) != len(set(lineages)) or any(not item for item in lineages):\n        raise ValueError("LINEAGE_IDS_MUST_BE_UNIQUE_AND_NONEMPTY")\n    if len(lineages) < 5:\n        raise ValueError("INSUFFICIENT_LINEAGES_FOR_THREE_SPLITS")\n    ordered = sorted(lineages, key=lambda item: hashlib.sha256(f"{seed}:{item}".encode()).digest())\n    train_end, validation_end = int(len(ordered) * 0.6), int(len(ordered) * 0.8)\n    groups = {name: split for split, names in zip(\n        SPLITS, (ordered[:train_end], ordered[train_end:validation_end], ordered[validation_end:]), strict=True)\n        for name in names}\n    payload = {"version": "lineage-split-v1", "seed": seed, "lineages": groups}\n    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False)\n    return {**payload, "sha256": hashlib.sha256(canonical.encode()).hexdigest()}\n\n\ndef valid_polygons(polygons) -> bool:\n    if not isinstance(polygons, list):\n        return False\n    for polygon in polygons:\n        if not isinstance(polygon, list) or len(polygon) < 3:\n            return False\n        for point in polygon:\n            if not isinstance(point, list) or len(point) != 2:\n                return False\n            if any(isinstance(v, bool) or not isinstance(v, (int, float))\n                   or not math.isfinite(v) or not 0 <= v <= 1 for v in point):\n                return False\n        area = abs(sum(polygon[i][0] * polygon[(i + 1) % len(polygon)][1]\n                       - polygon[(i + 1) % len(polygon)][0] * polygon[i][1]\n                       for i in range(len(polygon)))) / 2\n        if area <= 1e-8:\n            return False\n    return True\n\n\ndef validate_pairs(records: list[dict], split_manifest: dict) -> dict:\n    """Return explicit failures. UNKNOWN is masked, never interpreted as a negative."""\n    errors = []\n    seen_pairs = set()\n    hashes = defaultdict(set)\n    queries = defaultdict(set)\n    mapping = split_manifest.get("lineages", {})\n    counts = defaultdict(int)\n    for index, record in enumerate(records):\n        def fail(code):\n            errors.append({"row": index, "pair_id": record.get("pair_id"), "code": code})\n\n        required = ("pair_id", "query_id", "split", "query_lineage_id", "reference_lineage_id",\n                    "query_sha256", "reference_sha256", "source_kind", "source_ref", "license_ref",\n                    "annotator_id", "correspondence", "sufficiency", "observable_change",\n                    "change_polygons", "ignore_polygons", "reason_codes")\n        if any(key not in record for key in required):\n            fail("MISSING_REQUIRED_FIELDS")\n            continue\n        text_fields = set(required) - {"change_polygons", "ignore_polygons", "reason_codes"}\n        if any(not isinstance(record[key], str) or not record[key].strip() for key in text_fields):\n            fail("INVALID_TEXT_FIELD")\n            continue\n        pair_id, split = record["pair_id"], record["split"]\n        if pair_id in seen_pairs:\n            fail("DUPLICATE_PAIR_ID")\n        seen_pairs.add(pair_id)\n        if split not in SPLITS:\n            fail("INVALID_SPLIT")\n        counts[split] += 1\n        for side in ("query", "reference"):\n            if mapping.get(record[f"{side}_lineage_id"]) != split:\n                fail(f"{side.upper()}_LINEAGE_SPLIT_MISMATCH")\n            digest = record[f"{side}_sha256"]\n            if not SHA256.fullmatch(digest):\n                fail("INVALID_IMAGE_SHA256")\n            hashes[digest].add(split)\n        queries[record["query_id"]].add(split)\n        correspondence, sufficiency, change = (record[key] for key in (\n            "correspondence", "sufficiency", "observable_change"))\n        if correspondence not in CORRESPONDENCE or sufficiency not in SUFFICIENCY or change not in CHANGE:\n            fail("INVALID_LABEL_ENUM")\n        if correspondence != "MATCH" and sufficiency == "SUFFICIENT":\n            fail("SUFFICIENT_REQUIRES_MATCH")\n        if change != "UNKNOWN" and (correspondence != "MATCH" or sufficiency != "SUFFICIENT"):\n            fail("CHANGE_REQUIRES_SUFFICIENT_MATCH")\n        if correspondence in ("MATCH", "WRONG_REGION") and record["query_lineage_id"] != record["reference_lineage_id"]:\n            fail("SAME_OBJECT_LABEL_LINEAGE_MISMATCH")\n        if correspondence == "WRONG_OBJECT" and record["query_lineage_id"] == record["reference_lineage_id"]:\n            fail("WRONG_OBJECT_REQUIRES_DIFFERENT_LINEAGE")\n        if record["source_kind"] not in {"REAL", "SYNTHETIC", "SOFTWARE_FIXTURE"}:\n            fail("INVALID_SOURCE_KIND")\n        if not valid_polygons(record["change_polygons"]) or not valid_polygons(record["ignore_polygons"]):\n            fail("INVALID_NORMALIZED_POLYGON")\n        if record["change_polygons"] and change != "CHANGED":\n            fail("CHANGE_MASK_WITHOUT_CHANGED_LABEL")\n        if not isinstance(record["reason_codes"], list) or any(not isinstance(reason, str) or reason not in REASONS for reason in record["reason_codes"]):\n            fail("INVALID_REASON_CODES")\n    for digest, splits in hashes.items():\n        if len(splits) > 1:\n            errors.append({"code": "IMAGE_HASH_SPLIT_LEAKAGE", "sha256": digest, "splits": sorted(splits)})\n    for query_id, splits in queries.items():\n        if len(splits) > 1:\n            errors.append({"code": "QUERY_SPLIT_LEAKAGE", "query_id": query_id, "splits": sorted(splits)})\n    return {"gate": "PASS" if records and not errors else "FAIL", "rows": len(records),\n            "split_counts": dict(counts), "errors": errors,\n            "license_clearance": "SEPARATE_REVIEW_REQUIRED", "near_duplicate_audit": "SEPARATE_REVIEW_REQUIRED"}\n\n\ndef supervision(record: dict) -> dict:\n    match = record["correspondence"]\n    sufficient = record["sufficiency"]\n    return {\n        "same_region_target": float(match == "MATCH"),\n        "same_region_mask": match != "UNKNOWN",\n        "sufficiency_target": float(sufficient == "SUFFICIENT"),\n        "sufficiency_mask": match == "MATCH" and sufficient != "UNKNOWN",\n        "change_mask": match == "MATCH" and sufficient == "SUFFICIENT" and record["observable_change"] != "UNKNOWN",\n    }\n', 'ml/data/carton_views.py': '"""Procedural carton-face pretraining with ground-truth planar correspondence.\n\nNo Parcel3D pixels, external textures, brand assets or user photos are used.\nThis is synthetic pretraining, NOT a physical damage or real-box benchmark.\nFaces from a box share print style; distinct faces and nuisance views are hard\nnegatives. Labels derive from generation geometry, never SIFT/model predictions.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nfrom dataclasses import dataclass\n\nimport cv2\nimport numpy as np\n\nfrom ml.data.labels import locked_split\n\nVERSION = "carton-planar-pretrain-v1"\nSIZE = 384\nROI = [[2500, 2500], [7500, 2500], [7500, 7500], [2500, 7500]]\n\n\n@dataclass\nclass View:\n    id: str\n    face: int\n    image: np.ndarray\n    transform: np.ndarray\n    visible: np.ndarray\n    condition: str\n\n\ndef face_texture(box: int, face: int, size: int = SIZE) -> np.ndarray:\n    rng = np.random.default_rng(np.random.SeedSequence([26, box, face]))\n    base = np.array([185, 150, 104]) + rng.integers(-25, 26, 3)\n    image = np.clip(base + rng.normal(0, 3, (size, size, 1)), 0, 255).astype(np.uint8)\n    ink = tuple(int(x) for x in np.array([45, 38, 30]) + (box % 7) * 3)\n    # Common carton seam, variable tape and repeated generic packing marks.\n    seam = int(rng.integers(140, 240))\n    cv2.rectangle(image, (seam, 0), (seam + 26, size), (166, 133, 90), -1)\n    for _ in range(30):\n        x, y = rng.integers(8, size - 30, 2)\n        w, h = rng.integers(4, 45, 2)\n        cv2.rectangle(image, (int(x), int(y)), (int(x+w), int(y+h)), ink, int(rng.integers(1, 3)))\n    # Neutral synthetic label; no stable face/class identifier is printed.\n    x, y = (int(v) for v in rng.integers(20, 145, 2))\n    cv2.rectangle(image, (x, y), (x+160, y+90), (230, 225, 211), -1)\n    for index in range(45):\n        thickness = int(rng.integers(1, 4))\n        cv2.line(image, (x+8+index*3, y+12), (x+8+index*3, y+52), ink, thickness)\n    code = "".join(str(v) for v in rng.integers(0, 10, 10))\n    cv2.putText(image, code, (x+8, y+76), cv2.FONT_HERSHEY_SIMPLEX, 0.45, ink, 1, cv2.LINE_AA)\n    return image\n\n\ndef render_view(texture: np.ndarray, rng, identifier: str, face: int, condition: str) -> View:\n    source = np.float32([[0, 0], [SIZE-1, 0], [SIZE-1, SIZE-1], [0, SIZE-1]])\n    destination = source + rng.uniform(-38, 38, (4, 2)).astype(np.float32)\n    if condition == "partial":\n        destination[:, 0] -= 165\n    transform = cv2.getPerspectiveTransform(source, destination)\n    image = cv2.warpPerspective(texture, transform, (SIZE, SIZE), borderValue=(73, 79, 82))\n    visible = cv2.warpPerspective(np.ones((SIZE, SIZE), np.uint8), transform, (SIZE, SIZE), flags=cv2.INTER_NEAREST)\n    gain = rng.uniform(0.65, 1.3)\n    bias = rng.uniform(-16, 16)\n    image = np.clip(image.astype(np.float32) * gain + bias, 0, 255).astype(np.uint8)\n    if condition == "blur":\n        image = cv2.GaussianBlur(image, (27, 27), 7)\n    elif condition == "occlusion":\n        cv2.rectangle(image, (95, 120), (290, 275), (75, 83, 90), -1)\n        visible[120:276, 95:291] = 0\n    elif condition == "glare":\n        cv2.ellipse(image, (200, 200), (100, 60), 20, 0, 360, (255, 255, 255), -1)\n        cv2.ellipse(visible, (200, 200), (100, 60), 20, 0, 360, 0, -1)\n    return View(identifier, face, image, transform, visible, condition)\n\n\ndef box_views(box: int):\n    rng = np.random.default_rng(np.random.SeedSequence([2626, box]))\n    textures = [face_texture(box, face) for face in range(3)]\n    references = [render_view(textures[face], rng, f"b{box}:r{face}:{view}", face,\n                              "partial" if view == 1 else "clean")\n                  for face in range(3) for view in range(2)]\n    # Rotate face/condition independently of the locked split.\n    conditions = ["clean", "blur", "occlusion", "glare"]\n    queries = [render_view(textures[(box+i) % 3], rng, f"b{box}:q{i}", (box+i) % 3, condition)\n               for i, condition in enumerate(conditions)]\n    return queries, references\n\n\ndef pair_label(query: View, reference: View):\n    same = query.face == reference.face\n    overlap = 0.0\n    if same:\n        q_to_r = reference.transform @ np.linalg.inv(query.transform)\n        available = cv2.warpPerspective(reference.visible, np.linalg.inv(q_to_r), (SIZE, SIZE), flags=cv2.INTER_NEAREST)\n        roi = np.zeros((SIZE, SIZE), np.uint8)\n        cv2.fillPoly(roi, [np.rint(np.array(ROI)/10000*(SIZE-1)).astype(np.int32)], 1)\n        overlap = float(np.sum(available * query.visible * roi) / roi.sum())\n    sufficient = same and overlap >= 0.90 and query.condition != "blur"\n    return {"same_region": float(same), "sufficient": float(sufficient),\n            "ground_truth_overlap": overlap, "label_method": "KNOWN_GENERATOR_GEOMETRY"}\n\n\ndef corpus_split(boxes: int):\n    return locked_split([f"carton:{box}" for box in range(boxes)], seed=VERSION)\n\n\ndef image_hash(image):\n    return hashlib.sha256(image.tobytes()).hexdigest()\n', 'ml/features/__init__.py': '', 'ml/features/geometry.py': '"""Versioned pixel-only geometry features shared by training and inference."""\nfrom __future__ import annotations\n\nimport cv2\nimport numpy as np\n\nVERSION = "sift-pair-features-v2"\n\n\ndef supported_projection(points, homography, inlier_mask):\n    """Reject degenerate RANSAC output before deriving evidence features."""\n    if (homography is None or inlier_mask is None\n            or homography.shape != (3, 3) or not np.isfinite(homography).all()\n            or abs(np.linalg.det(homography)) < 1e-9\n            or inlier_mask.size != len(points)):\n        return None\n    known = inlier_mask.ravel().astype(bool)\n    if known.sum() < 4:\n        return None\n    selected = points[known]\n    denominator = selected @ homography[2, :2] + homography[2, 2]\n    if not np.isfinite(denominator).all() or (np.abs(denominator) < 1e-8).any():\n        return None\n    projected = cv2.perspectiveTransform(selected[:, None], homography)\n    if projected is None or not np.isfinite(projected).all():\n        return None\n    return known, projected[:, 0]\n\n\ndef describe(image, region=None):\n    rgb = np.asarray(image)\n    scale = min(1.0, 768 / max(rgb.shape[:2]))\n    rgb = cv2.resize(rgb, (round(rgb.shape[1]*scale), round(rgb.shape[0]*scale)))\n    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)\n    mask = np.full(gray.shape, 255, np.uint8)\n    if region is not None:\n        mask[:] = 0\n        points = np.rint(np.array(region)/10000*[rgb.shape[1]-1, rgb.shape[0]-1]).astype(np.int32)\n        cv2.fillPoly(mask, [points], 255)\n    kp, descriptors = cv2.SIFT_create(nfeatures=1600).detectAndCompute(gray, mask)\n    selected = mask > 0\n    laplacian = cv2.Laplacian(gray, cv2.CV_64F)\n    return {"keypoints": kp, "descriptors": descriptors, "shape": gray.shape, "mask": mask,\n            "sharpness": float(laplacian[selected].var()) if selected.any() else 0.,\n            "glare": float((gray[selected] > 248).mean()) if selected.any() else 1.}\n\n\ndef scalar_features(query, reference, cosine: float):\n    qd, rd = query["descriptors"], reference["descriptors"]\n    count, ratio, error, overlap, coverage = 0., 0., float("nan"), 0., 0.\n    if qd is not None and rd is not None and min(len(qd), len(rd)) >= 4:\n        good = [pair[0] for pair in cv2.BFMatcher().knnMatch(qd, rd, k=2)\n                if len(pair) == 2 and pair[0].distance < 0.72 * pair[1].distance]\n        unique = {}\n        for match in sorted(good, key=lambda match: match.distance):\n            unique.setdefault(match.trainIdx, match)\n        good = list(unique.values())\n        if len(good) >= 4:\n            qp = np.float32([query["keypoints"][match.queryIdx].pt for match in good])\n            rp = np.float32([reference["keypoints"][match.trainIdx].pt for match in good])\n            cv2.setRNGSeed(26)\n            H, inliers = cv2.findHomography(qp, rp, cv2.RANSAC, 4., maxIters=2000)\n            support = supported_projection(qp, H, inliers)\n            if support is not None:\n                known, projected = support\n                count, ratio = float(known.sum()), float(known.mean())\n                error = float(np.median(np.linalg.norm(projected-rp[known], axis=1))) / 768\n                h, w = query["shape"]\n                valid = cv2.warpPerspective(np.full(reference["shape"], 255, np.uint8), np.linalg.inv(H), (w, h))\n                roi = query["mask"] > 0\n                overlap = float((valid[roi] > 250).mean()) if roi.any() else 0.\n                hull = np.zeros((h, w), np.uint8)\n                if count >= 3:\n                    cv2.fillConvexPoly(hull, cv2.convexHull(qp[known]).astype(np.int32), 1)\n                coverage = float(hull[roi].mean()) if roi.any() else 0.\n    resolution = min(query["shape"]) / max(min(reference["shape"]), 1)\n    return np.array([cosine, min(count/100, 1), ratio, error, overlap, coverage,\n                     resolution, np.log1p(query["sharpness"]), np.log1p(reference["sharpness"]),\n                     query["glare"], reference["glare"], np.nan], dtype=np.float32)\n', 'ml/features/dino.py': '"""Frozen DINOv2 ViT-S/14. Original DINOv2, not the noncommercial X-ray variant."""\nfrom __future__ import annotations\n\nimport hashlib\nfrom pathlib import Path\n\nimport numpy as np\n\nDINO_REVISION = "7764ea0f912e53c92e82eb78a2a1631e92725fc8"\nWEIGHTS_URL = "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth"\n\n\nclass DinoEncoder:\n    def __init__(self, device="cpu", cache_dir=None):\n        import torch\n        self.torch, self.device = torch, device\n        if cache_dir is not None:\n            torch.hub.set_dir(str(cache_dir))\n        self.model = torch.hub.load(f"facebookresearch/dinov2:{DINO_REVISION}",\n                                   "dinov2_vits14", pretrained=True, trust_repo=True).to(device).eval()\n        weights = Path(torch.hub.get_dir()) / "checkpoints/dinov2_vits14_pretrain.pth"\n        self.weights_sha256 = hashlib.sha256(weights.read_bytes()).hexdigest()\n\n    def encode(self, images, batch_size=32):\n        import cv2\n        torch = self.torch\n        outputs = []\n        with torch.inference_mode():\n            for start in range(0, len(images), batch_size):\n                arrays = [cv2.resize(np.asarray(image), (224, 224)).astype(np.float32)/255\n                          for image in images[start:start+batch_size]]\n                batch = torch.from_numpy(np.stack(arrays).transpose(0, 3, 1, 2)).to(self.device)\n                mean = torch.tensor([.485, .456, .406], device=self.device)[None, :, None, None]\n                std = torch.tensor([.229, .224, .225], device=self.device)[None, :, None, None]\n                embeddings = self.model((batch-mean)/std)\n                embeddings = torch.nn.functional.normalize(embeddings, dim=1)\n                outputs.append(embeddings.cpu().numpy())\n        return np.concatenate(outputs).astype(np.float32)\n', 'ml/features/pair_features.py': '"""Versioned 1,560-D pair features; statistics may only be fit on training rows."""\nfrom __future__ import annotations\n\nimport numpy as np\n\nEMBEDDING_DIM = 384\nSCALAR_NAMES = ("cosine", "normalized_match_count", "inlier_ratio", "normalized_reprojection_error",\n                "overlap", "roi_coverage", "resolution_ratio", "query_blur", "reference_blur",\n                "query_glare", "reference_glare", "view_angle_difference")\nFEATURE_DIM = EMBEDDING_DIM * 4 + len(SCALAR_NAMES) * 2\nFEATURE_VERSION = "dinov2-s14-pair-v1"\n\n\nclass ScalarNormalizer:\n    def __init__(self, mean, scale):\n        self.mean = np.asarray(mean, dtype=np.float32)\n        self.scale = np.asarray(scale, dtype=np.float32)\n        if self.mean.shape != (12,) or self.scale.shape != (12,):\n            raise ValueError("EXPECTED_TWELVE_STATISTICS")\n        if not np.isfinite(self.mean).all() or not np.isfinite(self.scale).all() or (self.scale <= 0).any():\n            raise ValueError("INVALID_STATISTICS")\n\n    @classmethod\n    def fit(cls, scalars: np.ndarray, splits: list[str]):\n        values = np.asarray(scalars, dtype=np.float32)\n        if values.ndim != 2 or values.shape[1] != 12 or len(values) == 0:\n            raise ValueError("INVALID_SCALAR_MATRIX")\n        if len(splits) != len(values) or any(split != "train" for split in splits):\n            raise ValueError("NORMALIZER_TRAIN_ONLY")\n        means, scales = [], []\n        for column in values.T:\n            known = column[np.isfinite(column)]\n            means.append(float(known.mean()) if len(known) else 0.0)\n            scales.append(max(float(known.std()), 1e-6) if len(known) else 1.0)\n        return cls(means, scales)\n\n    def transform(self, values):\n        values = np.asarray(values, dtype=np.float32)\n        if values.ndim != 2 or values.shape[1] != 12:\n            raise ValueError("INVALID_SCALAR_MATRIX")\n        missing = ~np.isfinite(values)\n        normalized = (np.where(missing, self.mean, values) - self.mean) / self.scale\n        return normalized, missing.astype(np.float32)\n\n    def state(self):\n        return {"version": FEATURE_VERSION, "scalar_names": list(SCALAR_NAMES),\n                "mean": self.mean.tolist(), "scale": self.scale.tolist(), "fit_split": "train"}\n\n\ndef pair_features(query, reference, scalars, normalizer: ScalarNormalizer) -> np.ndarray:\n    query, reference = (np.asarray(array, dtype=np.float32) for array in (query, reference))\n    if query.ndim != 2 or query.shape[1] != EMBEDDING_DIM or query.shape != reference.shape:\n        raise ValueError("INVALID_EMBEDDING_SHAPE")\n    if not np.isfinite(query).all() or not np.isfinite(reference).all():\n        raise ValueError("NONFINITE_EMBEDDING")\n    query_norm, reference_norm = np.linalg.norm(query, axis=1, keepdims=True), np.linalg.norm(reference, axis=1, keepdims=True)\n    if (query_norm < 1e-8).any() or (reference_norm < 1e-8).any():\n        raise ValueError("ZERO_EMBEDDING")\n    query, reference = query / query_norm, reference / reference_norm\n    normalized, missing = normalizer.transform(scalars)\n    if len(normalized) != len(query):\n        raise ValueError("FEATURE_ROW_MISMATCH")\n    result = np.concatenate([query, reference, abs(query-reference), query*reference, normalized, missing], axis=1)\n    if result.shape[1] != FEATURE_DIM or not np.isfinite(result).all():\n        raise ValueError("FEATURE_ASSEMBLY_FAILED")\n    return result.astype(np.float32)\n', 'ml/train/__init__.py': '', 'ml/train/ranker.py': '"""Trainable heads for correspondence ranking and conditional evidence sufficiency.\n\nDefining/running these heads is not evidence of quality on real parcel images.\nNo network downloads or implicit pretrained weights occur in this module.\n"""\nfrom __future__ import annotations\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nINPUT_DIM = 1560\n\n\nclass EvidenceRanker(nn.Module):\n    def __init__(self, dropout: float = 0.1):\n        super().__init__()\n        self.shared = nn.Sequential(nn.Linear(INPUT_DIM, 256), nn.GELU(), nn.Dropout(dropout),\n                                    nn.Linear(256, 64), nn.GELU(), nn.Dropout(dropout))\n        self.ranking = nn.Linear(64, 1)\n        self.same_region = nn.Linear(64, 1)\n        self.sufficiency = nn.Linear(64, 1)\n\n    def forward(self, features):\n        if features.ndim != 2 or features.shape[1] != INPUT_DIM:\n            raise ValueError("RANKER_INPUT_SHAPE")\n        hidden = self.shared(features)\n        return {"ranking": self.ranking(hidden).squeeze(-1),\n                "same_region": self.same_region(hidden).squeeze(-1),\n                "sufficiency": self.sufficiency(hidden).squeeze(-1)}\n\n\ndef masked_bce(logits, targets, mask):\n    mask = mask.bool()\n    # Empty supervision contributes no gradient while preserving the graph.\n    return F.binary_cross_entropy_with_logits(logits[mask], targets[mask]) if mask.any() else logits.sum() * 0\n\n\ndef multitask_loss(outputs, labels, query_ids, ranking_weight=1.0, region_weight=1.0, sufficiency_weight=1.0):\n    count = len(outputs["ranking"])\n    if len(query_ids) != count or any(len(value) != count for value in labels.values()):\n        raise ValueError("LOSS_ROW_MISMATCH")\n    region = masked_bce(outputs["same_region"], labels["same_region_target"], labels["same_region_mask"])\n    sufficient = labels["sufficiency_mask"].bool() & labels["same_region_mask"].bool() & (labels["same_region_target"] == 1)\n    sufficiency = masked_bce(outputs["sufficiency"], labels["sufficiency_target"], sufficient)\n    ranking_terms = []\n    for query in dict.fromkeys(query_ids):\n        indices = torch.tensor([i for i, name in enumerate(query_ids) if name == query], device=outputs["ranking"].device)\n        known = labels["same_region_mask"][indices].bool()\n        positive = indices[known & (labels["same_region_target"][indices] == 1)]\n        negative = indices[known & (labels["same_region_target"][indices] == 0)]\n        if len(positive) and len(negative):\n            differences = outputs["ranking"][positive, None] - outputs["ranking"][negative][None, :]\n            ranking_terms.append(F.softplus(-differences).mean())\n    ranking = torch.stack(ranking_terms).mean() if ranking_terms else outputs["ranking"].sum() * 0\n    total = ranking_weight * ranking + region_weight * region + sufficiency_weight * sufficiency\n    return {"total": total, "ranking": ranking, "same_region": region, "sufficiency": sufficiency,\n            "ranked_queries": len(ranking_terms), "sufficiency_rows": int(sufficient.sum())}\n\n\ndef pipeline_check():\n    """CPU plumbing tests on fabricated tensors; not an accuracy experiment."""\n    torch.manual_seed(26)\n    model = EvidenceRanker(dropout=0)\n    inputs = torch.randn(8, INPUT_DIM)\n    labels = {\n        "same_region_target": torch.tensor([1, 0, 1, 0, 1, 0, 0, 0.], dtype=torch.float32),\n        "same_region_mask": torch.tensor([1, 1, 1, 1, 1, 1, 0, 0], dtype=torch.bool),\n        "sufficiency_target": torch.tensor([1, 0, 0, 0, 1, 0, 0, 0.], dtype=torch.float32),\n        "sufficiency_mask": torch.tensor([1, 0, 1, 0, 1, 0, 0, 0], dtype=torch.bool),\n    }\n    queries = ["q1", "q1", "q2", "q2", "q3", "q3", "q4", "q4"]\n    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)\n    initial = float(multitask_loss(model(inputs), labels, queries)["total"].detach())\n    for _ in range(12):\n        optimizer.zero_grad()\n        losses = multitask_loss(model(inputs), labels, queries)\n        losses["total"].backward()\n        assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)\n        optimizer.step()\n    final = float(multitask_loss(model(inputs), labels, queries)["total"].detach())\n    assert final < initial, "OPTIMIZER_DID_NOT_REDUCE_TOY_LOSS"\n    unknown = {key: value.clone() for key, value in labels.items()}\n    unknown["same_region_mask"][:] = False\n    unknown["sufficiency_mask"][:] = False\n    assert multitask_loss(model(inputs), unknown, queries)["total"].item() == 0\n    with torch.no_grad():\n        first = model(inputs)\n        restored = EvidenceRanker(dropout=0)\n        restored.load_state_dict(model.state_dict())\n        for key, value in restored(inputs).items():\n            torch.testing.assert_close(value, first[key])\n    return {"gate": "PASS", "test_kind": "FABRICATED_TENSOR_PIPELINE_CHECK",\n            "real_data_accuracy_measured": False, "trained_release_checkpoint": False,\n            "parameter_count": sum(p.numel() for p in model.parameters()),\n            "torch_version": torch.__version__, "device": "cpu", "optimizer_steps": 12,\n            "initial_toy_loss": initial, "final_toy_loss": final}\n', 'ml/train/synthetic_ranker.py': '"""Train/evaluate a real image-feature model on explicitly synthetic carton views.\n\nObject-disjoint train/model-selection/calibration/test sets. The final test is\nopened only after seed selection, temperature fitting and threshold selection.\nThis experiment cannot authorize real-world evidence acceptance on its own.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport platform\nimport time\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nfrom PIL import Image, ImageDraw\nfrom torch.nn import functional as F\n\nfrom ml.data.carton_views import ROI, VERSION, box_views, corpus_split, image_hash, pair_label\nfrom ml.features.dino import DINO_REVISION, DinoEncoder\nfrom ml.features.geometry import describe, scalar_features\nfrom ml.features.pair_features import FEATURE_VERSION, ScalarNormalizer, pair_features\nfrom ml.train.ranker import EvidenceRanker, multitask_loss\n\n\ndef dump(path, value):\n    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")\n\n\ndef sigmoid(values):\n    return 1 / (1 + np.exp(-np.clip(values, -50, 50)))\n\n\ndef temperatures(logits, targets):\n    options = np.geomspace(.25, 8., 80)\n    losses = []\n    for temp in options:\n        p = np.clip(sigmoid(logits / temp), 1e-6, 1-1e-6)\n        losses.append(float(-(targets*np.log(p) + (1-targets)*np.log(1-p)).mean()))\n    return float(options[np.argmin(losses)])\n\n\ndef metrics(prediction, rows, threshold=None, temperatures_=(1., 1.)):\n    ranking = prediction["ranking"].reshape(-1, 6)\n    target = np.array([row["same_region"] for row in rows]).reshape(-1, 6)\n    suff = np.array([row["sufficient"] for row in rows]).reshape(-1, 6)\n    best = ranking.argmax(axis=1)\n    indices = np.arange(len(best))\n    top_correct = target[indices, best] == 1\n    safe = suff[indices, best] == 1\n    region_probability = sigmoid(prediction["same_region"] / temperatures_[0]).reshape(-1, 6)\n    suff_probability = sigmoid(prediction["sufficiency"] / temperatures_[1]).reshape(-1, 6)\n    accepted = ((region_probability[indices, best] >= threshold) &\n                (suff_probability[indices, best] >= threshold)) if threshold is not None else np.ones(len(best), bool)\n    groups = np.array([row["box_id"] for row in rows[::6]])\n    rng = np.random.default_rng(2626)\n    # Cluster bootstrap resamples entire boxes, preserving correlated views.\n    unique = np.unique(groups)\n    samples = []\n    if len(unique):\n        grouped = {box: np.flatnonzero(groups == box) for box in unique}\n        for _ in range(400):\n            sample = np.concatenate([grouped[box] for box in rng.choice(unique, len(unique))])\n            samples.append(float(top_correct[sample].mean()))\n    result = {"queries": len(best), "boxes": len(unique), "top1_correspondence": float(top_correct.mean()),\n              "top1_box_bootstrap_ci95": np.quantile(samples, [.025, .975]).tolist(),\n              "accepted_queries": int(accepted.sum()), "coverage": float(accepted.mean()),\n              "unsafe_accepts": int((accepted & ~safe).sum()),\n              "accepted_precision": float(safe[accepted].mean()) if accepted.any() else None,\n              "sufficient_query_recall": float((accepted & safe).sum()/max((suff.max(axis=1) == 1).sum(), 1))}\n    result["by_condition"] = {}\n    conditions = np.array([row["condition"] for row in rows[::6]])\n    for condition in np.unique(conditions):\n        selected = conditions == condition\n        result["by_condition"][condition] = {"queries": int(selected.sum()),\n            "top1": float(top_correct[selected].mean()), "accepted": int(accepted[selected].sum()),\n            "unsafe_accepts": int((accepted[selected] & ~safe[selected]).sum())}\n    return result\n\n\ndef train_candidate(features, labels, train_idx, select_idx, seed, device, epochs=24):\n    torch.manual_seed(seed)\n    model = EvidenceRanker().to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=.01)\n    rng = np.random.default_rng(seed)\n    best_loss, best_state, logs = float("inf"), None, []\n    train_queries = train_idx.reshape(-1, 6)\n    for epoch in range(epochs):\n        model.train()\n        losses = []\n        order = rng.permutation(len(train_queries))\n        for start in range(0, len(order), 16):\n            idx = train_queries[order[start:start+16]].reshape(-1)\n            x = torch.from_numpy(features[idx]).to(device)\n            batch_labels = {key: torch.from_numpy(value[idx]).to(device) for key, value in labels.items()}\n            output = model(x)\n            query_ids = [f"q{i//6}" for i in range(len(idx))]\n            loss = multitask_loss(output, batch_labels, query_ids)["total"]\n            optimizer.zero_grad()\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.)\n            optimizer.step()\n            losses.append(float(loss.detach()))\n        model.eval()\n        with torch.inference_mode():\n            output = model(torch.from_numpy(features[select_idx]).to(device))\n            known = torch.from_numpy(labels["same_region_target"][select_idx]).to(device)\n            sufficient = torch.from_numpy(labels["sufficiency_target"][select_idx]).to(device)\n            positive = known == 1\n            selection_loss = float(F.binary_cross_entropy_with_logits(output["same_region"], known) +\n                                   F.binary_cross_entropy_with_logits(output["sufficiency"][positive], sufficient[positive]))\n        logs.append({"epoch": epoch+1, "training_loss": float(np.mean(losses)), "selection_loss": selection_loss})\n        if selection_loss < best_loss:\n            best_loss = selection_loss\n            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}\n    model.load_state_dict(best_state)\n    model.eval()\n    print(json.dumps({"seed_complete": seed, "selection_loss": best_loss}), flush=True)\n    return model, best_loss, logs\n\n\ndef run(output: Path, boxes=300, epochs=24):\n    output.mkdir(parents=True, exist_ok=True)\n    started = time.monotonic()\n    cv2.setNumThreads(1)\n    torch.set_num_threads(2)\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    split = corpus_split(boxes)\n    dump(output / "split.json", split)\n    encoder = DinoEncoder(device=device, cache_dir=Path("/tmp/kienchung-dino"))\n    rows, query_embeddings, reference_embeddings, scalars = [], [], [], []\n    gallery = []\n    for box in range(boxes):\n        queries, references = box_views(box)\n        all_views = queries + references\n        embeddings = encoder.encode([view.image for view in all_views])\n        descriptions = [describe(view.image, ROI if index < 4 else None) for index, view in enumerate(all_views)]\n        for qi, query in enumerate(queries):\n            for ri, reference in enumerate(references):\n                qembedding, rembedding = embeddings[qi], embeddings[4+ri]\n                scalar = scalar_features(descriptions[qi], descriptions[4+ri], float(qembedding @ rembedding))\n                rows.append({"pair_id": f"{query.id}:{reference.id}", "query_id": query.id,\n                    "box_id": f"carton:{box}", "split": split["lineages"][f"carton:{box}"],\n                    "condition": query.condition, "query_sha256": image_hash(query.image),\n                    "reference_sha256": image_hash(reference.image), **pair_label(query, reference)})\n                query_embeddings.append(qembedding)\n                reference_embeddings.append(rembedding)\n                scalars.append(scalar)\n        if box < 4:\n            gallery.extend(queries + references[:2])\n        if (box+1) % 25 == 0:\n            print(json.dumps({"images_processed": (box+1)*10, "pairs": len(rows), "seconds": time.monotonic()-started}), flush=True)\n    scalar_array = np.stack(scalars)\n    splits = np.array([row["split"] for row in rows])\n    train_idx = np.flatnonzero(splits == "train")\n    validation_boxes = sorted({row["box_id"] for row in rows if row["split"] == "validation"})\n    selection_boxes = set(validation_boxes[::2])\n    selection_idx = np.array([i for i, row in enumerate(rows) if row["box_id"] in selection_boxes])\n    calibration_idx = np.array([i for i, row in enumerate(rows) if row["split"] == "validation" and row["box_id"] not in selection_boxes])\n    test_idx = np.flatnonzero(splits == "test")\n    normalizer = ScalarNormalizer.fit(scalar_array[train_idx], ["train"]*len(train_idx))\n    features = pair_features(np.stack(query_embeddings), np.stack(reference_embeddings), scalar_array, normalizer)\n    labels = {"same_region_target": np.array([row["same_region"] for row in rows], np.float32),\n              "same_region_mask": np.ones(len(rows), bool),\n              "sufficiency_target": np.array([row["sufficient"] for row in rows], np.float32),\n              "sufficiency_mask": np.array([row["same_region"] == 1 for row in rows])}\n    candidates, training_logs = [], {}\n    for seed in (26, 27, 28):\n        model, loss, log = train_candidate(features, labels, train_idx, selection_idx, seed, device, epochs)\n        candidates.append((loss, seed, model.cpu()))\n        training_logs[str(seed)] = log\n    _, chosen_seed, model = min(candidates, key=lambda item: item[0])\n    model = model.to(device)\n    with torch.inference_mode():\n        calibration = {key: value.cpu().numpy() for key, value in model(torch.from_numpy(features[calibration_idx]).to(device)).items()}\n    positive = labels["same_region_target"][calibration_idx] == 1\n    temp = (temperatures(calibration["same_region"], labels["same_region_target"][calibration_idx]),\n            temperatures(calibration["sufficiency"][positive], labels["sufficiency_target"][calibration_idx][positive]))\n    calibration_rows = [rows[i] for i in calibration_idx]\n    chosen_threshold, best_recall = 1., -1.\n    for threshold in (.70, .80, .85, .90, .95, .975, .99):\n        measured = metrics(calibration, calibration_rows, threshold, temp)\n        if measured["unsafe_accepts"] == 0 and measured["sufficient_query_recall"] > best_recall:\n            chosen_threshold, best_recall = threshold, measured["sufficient_query_recall"]\n    # Test is evaluated only now; do not select checkpoints, thresholds or seeds here.\n    with torch.inference_mode():\n        test_predictions = {key: value.cpu().numpy() for key, value in model(torch.from_numpy(features[test_idx]).to(device)).items()}\n    test_rows = [rows[i] for i in test_idx]\n    test_metrics = metrics(test_predictions, test_rows, chosen_threshold, temp)\n    baselines = {}\n    for name, score in {"dino_cosine": scalar_array[test_idx, 0],\n                        "sift_geometry": scalar_array[test_idx, 1]*scalar_array[test_idx, 2]*scalar_array[test_idx, 5]}.items():\n        baseline = {"ranking": score, "same_region": score, "sufficiency": score}\n        baseline_metrics = metrics(baseline, test_rows)\n        baselines[name] = {key: baseline_metrics[key] for key in ("queries", "boxes", "top1_correspondence", "top1_box_bootstrap_ci95")}\n    weights = {key: value.detach().cpu().numpy() for key, value in model.state_dict().items()}\n    np.savez_compressed(output / "ranker.npz", **weights)\n    np.savez_compressed(output / "test-predictions.npz", **test_predictions, indices=test_idx)\n    # Probe enables independently checking the NumPy deployment implementation.\n    probe_idx = selection_idx[:12]\n    with torch.inference_mode():\n        probe_outputs = {key: value.cpu().numpy() for key, value in model(torch.from_numpy(features[probe_idx]).to(device)).items()}\n    np.savez_compressed(output / "export-parity.npz", features=features[probe_idx], **probe_outputs)\n    dump(output / "normalizer.json", normalizer.state())\n    dump(output / "training-log.json", training_logs)\n    (output / "labels.jsonl").write_text("\\n".join(json.dumps(row) for row in rows), encoding="utf-8")\n    sheet = Image.new("RGB", (6*192, 4*220), "white")\n    draw = ImageDraw.Draw(sheet)\n    for index, view in enumerate(gallery):\n        x, y = index % 6 * 192, index // 6 * 220\n        sheet.paste(Image.fromarray(view.image).resize((192, 192)), (x, y))\n        draw.text((x+4, y+196), f"{view.id} {view.condition}", fill="black")\n    sheet.save(output / "sample-gallery.jpg", quality=90)\n    card = {"model_version": "synthetic-dino-ranker-v1", "experiment_kind": "SYNTHETIC_IMAGE_PRETRAINING",\n        "real_image_validation": "NOT_RUN", "release_gate": "HOLD_REAL_BOX_VALIDATION",\n        "dataset": VERSION, "boxes": boxes, "images": boxes*10, "pairs": len(rows),\n        "split_hash": split["sha256"], "selected_seed": chosen_seed,\n        "selection_boxes": sorted(selection_boxes),\n        "calibration_boxes": sorted(set(validation_boxes)-selection_boxes),\n        "temperatures": list(temp), "threshold": chosen_threshold,\n        "feature_version": FEATURE_VERSION, "dino_revision": DINO_REVISION,\n        "dino_weights_sha256": encoder.weights_sha256,\n        "checkpoint_sha256": hashlib.sha256((output/"ranker.npz").read_bytes()).hexdigest(),\n        "normalizer_sha256": hashlib.sha256((output/"normalizer.json").read_bytes()).hexdigest(),\n        "test": test_metrics, "baselines": baselines,\n        "seed_selection_losses": {str(seed): loss for loss, seed, _ in candidates},\n        "runtime": {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__,\n                    "opencv": cv2.__version__, "device": device, "seconds": time.monotonic()-started},\n        "limitations": ["Planar synthetic faces only; no physical dent model or damage masks.",\n                        "No real customer photos or human sufficiency annotation.",\n                        "Calibration applies only to this synthetic distribution.",\n                        "Not permitted to authorize settlement or replace human review."]}\n    dump(output / "model-card.json", card)\n    print(json.dumps(card), flush=True)\n    return card\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--boxes", type=int, default=300)\n    parser.add_argument("--epochs", type=int, default=24)\n    args = parser.parse_args()\n    run(args.output, args.boxes, args.epochs)\n'}
bundle = Path('/kaggle/working/_src')
for name, source in SOURCES.items():
    path = bundle / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding='utf-8')
sys.path.insert(0, str(bundle))


In [ ]:
from ml.train.synthetic_ranker import run
result = run(Path('/kaggle/working/training'), boxes=300, epochs=24)
Path('/kaggle/working/training/source-hashes.json').write_text(json.dumps({'ml/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/data/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/data/labels.py': 'ee7341c80f127e0f5d9cfa28ad9b58bb7a3ddff12e8dec893c65802d8e963bef', 'ml/data/carton_views.py': '521bef9e87ec14ec4c546ab58face8b5696310cd449d3101b962c522c1e9b66c', 'ml/features/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/features/geometry.py': 'c08b867e13ca331207029af95a64959bf8bd7ab3cf8ebbad22290a373347b00b', 'ml/features/dino.py': '799f5a6c1ee8cdf43ef8c9c3428477f009dcdbe30392c02589ae73ee577d6493', 'ml/features/pair_features.py': 'bf43387b92a040ef1b2e2528836b73eca79de20eb02ea31962d5180efade217e', 'ml/train/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/train/ranker.py': '9db0f01016594b3a99217eb86228073dc015722bc251c61fa623ea4a67bfd80f', 'ml/train/synthetic_ranker.py': '9f7529036710b5105d6e81e54e581b7b526b19119b3327a3924b54a78b9137a1'}, indent=2))
